## Introduction
In this guide, we will walk you through building a powerful semantic search engine using Couchbase as the backend database, [Azure OpenAI](https://azure.microsoft.com/) as the AI-powered embedding and language model provider. Semantic search goes beyond simple keyword matching by understanding the context and meaning behind the words in a query, making it an essential tool for applications that require intelligent information retrieval. This tutorial is designed to be beginner-friendly, with clear, step-by-step instructions that will equip you with the knowledge to create a fully functional semantic search system using the Search Vector Index from scratch. Alternatively if you want to perform semantic search using the Hyperscale or Composite Vector Index, please take a look at [this.](https://developer.couchbase.com/tutorial-azure-openai-couchbase-rag-with-hyperscale-or-composite-vector-index/)

## How to run this tutorial

This tutorial is available as a Jupyter Notebook (`.ipynb` file) that you can run interactively. You can access the original notebook [here](https://github.com/couchbase-examples/vector-search-cookbook/blob/main/azure/search_based/RAG_with_Couchbase_and_AzureOpenAI.ipynb).

You can either download the notebook file and run it on [Google Colab](https://colab.research.google.com/) or run it on your system by setting up the Python environment.

## Before you start

### Get Credentials for Azure OpenAI

Please follow the [instructions](https://learn.microsoft.com/en-us/azure/ai-services/openai/reference) to generate the Azure OpenAI credentials.

### Create and Deploy Your Free Tier Operational cluster on Capella

To get started with Couchbase Capella, create an account and use it to deploy a forever free tier operational cluster. This account provides you with a environment where you can explore and learn about Capella with no time constraint.

To know more, please follow the [instructions](https://docs.couchbase.com/cloud/get-started/create-account.html).

#### Couchbase Capella Configuration

When running Couchbase using [Capella](https://cloud.couchbase.com/sign-in), the following prerequisites need to be met.

* Create the [database credentials](https://docs.couchbase.com/cloud/clusters/manage-database-users.html) to access the travel-sample bucket (Read and Write) used in the application.
* [Allow access](https://docs.couchbase.com/cloud/clusters/allow-ip-address.html) to the Cluster from the IP on which the application is running.

## Setting the Stage: Installing Necessary Libraries
To build our semantic search engine, we need a robust set of tools. The libraries we install handle everything from connecting to databases to performing complex machine learning tasks. Each library has a specific role: Couchbase libraries manage database operations, LangChain handles AI model integrations, and Azure OpenAI provides advanced AI models for generating embeddings and understanding natural language. By setting up these libraries, we ensure our environment is equipped to handle the data-intensive and computationally complex tasks required for semantic search.

In [1]:
!pip install datasets==3.5.0 langchain-couchbase==0.3.0 langchain-openai==0.3.13 python-dotenv

  Using cached langchain_couchbase-0.3.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_openai-0.3.13-py3-none-any.whl.metadata (2.3 kB)
Using cached langchain_couchbase-0.3.0-py3-none-any.whl (22 kB)
Using cached langchain_openai-0.3.13-py3-none-any.whl (61 kB)
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 0.3.32
    Uninstalling langchain-openai-0.3.32:
      Successfully uninstalled langchain-openai-0.3.32
  Attempting uninstall: langchain-couchbase
    Found existing installation: langchain-couchbase 0.5.0
    Uninstalling langchain-couchbase-0.5.0:
      Successfully uninstalled langchain-couchbase-0.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-couchbase]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Importing Necessary Libraries
The script starts by importing a series of libraries required for various tasks, including handling JSON, logging, time tracking, Couchbase connections, embedding generation, and dataset loading. These libraries provide essential functions for working with data, managing database connections, and processing machine learning models.

In [2]:
import os
import getpass
import json
import logging
import sys
import time
import random
from datetime import timedelta
from uuid import uuid4

from couchbase.auth import PasswordAuthenticator
from couchbase.cluster import Cluster
from couchbase.exceptions import (
    CouchbaseException,
    InternalServerFailureException,
    QueryIndexAlreadyExistsException,
)
from couchbase.management.search import SearchIndex
from couchbase.options import ClusterOptions
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_core.globals import set_llm_cache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_couchbase.cache import CouchbaseCache
from langchain_couchbase.vectorstores import CouchbaseSearchVectorStore
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from tqdm import tqdm
from dotenv import load_dotenv

/Users/viraj.agarwal/Tasks/2026/Task1.5/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup Logging
Logging is configured to track the progress of the script and capture any errors or warnings. This is crucial for debugging and understanding the flow of execution. The logging output includes timestamps, log levels (e.g., INFO, ERROR), and messages that describe what is happening in the script.


In [3]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)

def retry_with_backoff(func, max_retries=3, base_delay=1, max_delay=10):
    """Execute function with exponential backoff retry for rate limiting."""
    for attempt in range(max_retries):
        try:
            return func()
        except InternalServerFailureException as e:
            if "query request rejected" in str(e) or "429" in str(e):
                if attempt < max_retries - 1:
                    delay = min(base_delay * (2 ** attempt) + random.uniform(0, 1), max_delay)
                    logging.warning(f"Rate limited. Retrying in {delay:.2f}s (attempt {attempt + 1}/{max_retries})")
                    time.sleep(delay)
                else:
                    raise
            else:
                raise
        except Exception as e:
            raise
    return None

## Loading Sensitive Information
In this section, we prompt the user to input essential configuration settings needed. These settings include sensitive information like API keys, database credentials, and specific configuration names. Instead of hardcoding these details into the script, we request the user to provide them at runtime, ensuring flexibility and security.

The script also validates that all required inputs are provided, raising an error if any crucial information is missing. This approach ensures that your integration is both secure and correctly configured without hardcoding sensitive information, enhancing the overall security and maintainability of your code.

In [4]:
load_dotenv()  # Loads variables from .env into environment

AZURE_OPENAI_KEY = os.getenv('AZURE_OPENAI_KEY') or getpass.getpass('Enter your Azure OpenAI Key: ')
AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT') or input('Enter your Azure OpenAI Endpoint: ')
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT') or input('Enter your Azure OpenAI Embedding Deployment: ')
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT') or input('Enter your Azure OpenAI Chat Deployment: ')

CB_HOST = os.getenv('CB_HOST') or input('Enter your Couchbase host (default: couchbase://localhost): ') or 'couchbase://localhost'
CB_USERNAME = os.getenv('CB_USERNAME') or input('Enter your Couchbase username (default: Administrator): ') or 'Administrator'
CB_PASSWORD = os.getenv('CB_PASSWORD') or getpass.getpass('Enter your Couchbase password (default: password): ') or 'password'
CB_BUCKET_NAME = os.getenv('CB_BUCKET_NAME') or input('Enter your Couchbase bucket name (default: vector-search-testing): ') or 'vector-search-testing'
INDEX_NAME = os.getenv('INDEX_NAME') or input('Enter your index name (default: vector_search_azure): ') or 'vector_search_azure'
SCOPE_NAME = os.getenv('SCOPE_NAME') or input('Enter your scope name (default: shared): ') or 'shared'
COLLECTION_NAME = os.getenv('COLLECTION_NAME') or input('Enter your collection name (default: azure): ') or 'azure'
CACHE_COLLECTION = os.getenv('CACHE_COLLECTION') or input('Enter your cache collection name (default: cache): ') or 'cache'

# Check if the variables are correctly loaded
if not all([AZURE_OPENAI_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_EMBEDDING_DEPLOYMENT, AZURE_OPENAI_CHAT_DEPLOYMENT]):
    raise ValueError("Missing required Azure OpenAI variables")

## Connecting to the Couchbase Cluster
Connecting to a Couchbase cluster is the foundation of our project. Couchbase will serve as our primary data store, handling all the storage and retrieval operations required for our semantic search engine. By establishing this connection, we enable our application to interact with the database, allowing us to perform operations such as storing embeddings, querying data, and managing collections. This connection is the gateway through which all data will flow, so ensuring it's set up correctly is paramount.



In [5]:
try:
    auth = PasswordAuthenticator(CB_USERNAME, CB_PASSWORD)
    options = ClusterOptions(auth)
    options.apply_profile('wan_development')
    cluster = Cluster(CB_HOST, options)
    cluster.wait_until_ready(timedelta(seconds=5))
    logging.info("Successfully connected to Couchbase")
except Exception as e:
    raise ConnectionError(f"Failed to connect to Couchbase: {str(e)}")

2026-04-06 14:52:41,588 - INFO - Successfully connected to Couchbase


## Setting Up Collections in Couchbase
In Couchbase, data is organized in buckets, which can be further divided into scopes and collections. Think of a collection as a table in a traditional SQL database. Before we can store any data, we need to ensure that our collections exist. If they don't, we must create them. This step is important because it prepares the database to handle the specific types of data our application will process. By setting up collections, we define the structure of our data storage, which is essential for efficient data retrieval and management.

Moreover, setting up collections allows us to isolate different types of data within the same bucket, providing a more organized and scalable data structure. This is particularly useful when dealing with large datasets, as it ensures that related data is stored together, making it easier to manage and query.

In [ ]:
def setup_collection(cluster, bucket_name, scope_name, collection_name):
    try:
        bucket = cluster.bucket(bucket_name)
        bucket_manager = bucket.collections()

        # Check if collection exists, create if it doesn't
        collections = bucket_manager.get_all_scopes()
        collection_exists = any(
            scope.name == scope_name and collection_name in [col.name for col in scope.collections]
            for scope in collections
        )

        if not collection_exists:
            logging.info(f"Collection '{collection_name}' does not exist. Creating it...")
            bucket_manager.create_collection(scope_name, collection_name)
            logging.info(f"Collection '{collection_name}' created successfully.")
        else:
            logging.info(f"Collection '{collection_name}' already exists.Skipping creation.")

        collection = bucket.scope(scope_name).collection(collection_name)
        time.sleep(2)  # Give the collection time to be ready for queries

        # Ensure primary index exists
        try:
            cluster.query(f"CREATE PRIMARY INDEX IF NOT EXISTS ON `{bucket_name}`.`{scope_name}`.`{collection_name}`").execute()
            logging.info("Primary index present or created successfully.")
        except Exception as e:
            logging.warning(f"Error creating primary index: {str(e)}")

        # Clear all documents in the collection
        try:
            query = f"DELETE FROM `{bucket_name}`.`{scope_name}`.`{collection_name}`"
            cluster.query(query).execute()
            logging.info("All documents cleared from the collection.")
        except Exception as e:
            logging.warning(f"Error while clearing documents: {str(e)}. The collection might be empty.")

        return collection
    except Exception as e:
        raise RuntimeError(f"Error setting up collection: {str(e)}")

setup_collection(cluster, CB_BUCKET_NAME, SCOPE_NAME, COLLECTION_NAME)
setup_collection(cluster, CB_BUCKET_NAME, SCOPE_NAME, CACHE_COLLECTION)

## Loading Couchbase Vector Search Index

Semantic search requires an efficient way to retrieve relevant documents based on a user's query. This is where the Couchbase **Vector Search Index** comes into play. In this step, we load the Vector Search Index definition from a JSON file, which specifies how the index should be structured. This includes the fields to be indexed, the dimensions of the vectors, and other parameters that determine how the search engine processes queries based on vector similarity.

For more information on creating a vector search index, please follow the [instructions](https://docs.couchbase.com/cloud/vector-search/create-vector-search-index-ui.html).


In [7]:
# If you are running this script locally (not in Google Colab), uncomment the following line
# and provide the path to your index definition file.

index_definition_path = './azure_index.json'  # Local setup: specify your file path here

# If you are running in Google Colab, use the following code to upload the index definition file
# from google.colab import files
# print("Upload your index definition file")
# uploaded = files.upload()
# index_definition_path = list(uploaded.keys())[0]

try:
    with open(index_definition_path, 'r') as file:
        index_definition = json.load(file)
    # Update the index definition with the current bucket name and scope/collection
    index_definition['sourceName'] = CB_BUCKET_NAME
    # Update the type mapping to use current scope and collection names
    if 'params' in index_definition and 'mapping' in index_definition['params']:
        if 'types' in index_definition['params']['mapping']:
            types = index_definition['params']['mapping']['types']
            # Replace old scope.collection with new ones
            for old_key in list(types.keys()):
                if '.' in old_key:
                    new_key = f"{SCOPE_NAME}.{COLLECTION_NAME}"
                    types[new_key] = types.pop(old_key)
except Exception as e:
    raise ValueError(f"Error loading index definition from {index_definition_path}: {str(e)}")

## Creating or Updating Search Indexes

With the index definition loaded, the next step is to create or update the **Vector Search Index** in Couchbase. This step is crucial because it optimizes our database for vector similarity search operations, allowing us to perform searches based on the semantic content of documents rather than just keywords. By creating or updating a Vector Search Index, we enable our search engine to handle complex queries that involve finding semantically similar documents using vector embeddings, which is essential for a robust semantic search engine.

In [8]:
try:
    scope_index_manager = cluster.bucket(CB_BUCKET_NAME).scope(SCOPE_NAME).search_indexes()

    # Check if index already exists
    existing_indexes = scope_index_manager.get_all_indexes()
    index_name = index_definition["name"]

    if index_name in [index.name for index in existing_indexes]:
        logging.info(f"Index '{index_name}' found")
    else:
        logging.info(f"Creating new index '{index_name}'...")

    # Create SearchIndex object from JSON definition
    search_index = SearchIndex.from_json(index_definition)

    # Upsert the index (create if not exists, update if exists)
    scope_index_manager.upsert_index(search_index)
    logging.info(f"Index '{index_name}' successfully created/updated.")

except QueryIndexAlreadyExistsException:
    logging.info(f"Index '{index_name}' already exists. Skipping creation/update.")

except InternalServerFailureException as e:
    error_message = str(e)
    logging.error(f"InternalServerFailureException raised: {error_message}")

    try:
        # Accessing the response_body attribute from the context
        error_context = e.context
        response_body = error_context.response_body
        if response_body:
            error_details = json.loads(response_body)
            error_message = error_details.get('error', '')

            if "collection: 'azure' doesn't belong to scope: 'shared'" in error_message:
                raise ValueError("Collection 'azure' does not belong to scope 'shared'. Please check the collection and scope names.")

    except ValueError as ve:
        logging.error(str(ve))
        raise

    except Exception as json_error:
        logging.error(f"Failed to parse the error message: {json_error}")
        raise RuntimeError(f"Internal server error while creating/updating search index: {error_message}")

2026-04-06 14:52:51,019 - INFO - Creating new index 'vector_search_azure'...
2026-04-06 14:52:51,120 - INFO - Index 'vector_search_azure' successfully created/updated.


## Load the BBC News Dataset
To build a search engine, we need data to search through. We use the BBC News dataset from RealTimeData, which provides real-world news articles across multiple topics and time periods. Loading the dataset is a crucial step because it provides the raw material that our search engine will work with.

The BBC News dataset allows us to work with authentic news content, enabling us to build and test retrieval quality on realistic long-form articles.

In [9]:
try:
    news_dataset = load_dataset("RealTimeData/bbc_news_alltime", "2024-12", split="train")
    news_articles = news_dataset["content"]
    unique_news_articles = list({article for article in news_articles if article})
    logging.info(f"Successfully loaded BBC News dataset with {len(news_dataset)} rows")
    logging.info(f"Prepared {len(unique_news_articles)} unique BBC news articles for ingestion")
except Exception as e:
    raise ValueError(f"Error loading BBC News dataset: {str(e)}")

2026-04-06 14:52:54,685 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/RealTimeData/bbc_news_alltime/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-06 14:52:54,904 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/RealTimeData/bbc_news_alltime/8dd1ecdc92ac43f9c04a3da3e945537dbb08179b/README.md "HTTP/1.1 200 OK"
2026-04-06 14:52:55,205 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/RealTimeData/bbc_news_alltime/resolve/8dd1ecdc92ac43f9c04a3da3e945537dbb08179b/bbc_news_alltime.py "HTTP/1.1 404 Not Found"
2026-04-06 14:52:56,188 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/RealTimeData/bbc_news_alltime/RealTimeData/bbc_news_alltime.py "HTTP/1.1 404 Not Found"
2026-04-06 14:52:56,602 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/RealTimeData/bbc_news_alltime/revision/8dd1ecdc92ac43f9c04a3da3e945537dbb08179b "HTTP/1.1 200 OK"
2026-04-06 14:52:56,922 - IN

## Creating Azure OpenAI Embeddings
Embeddings are at the heart of semantic search. They are numerical representations of text that capture the semantic meaning of the words and phrases. Unlike traditional keyword-based search, which looks for exact matches, embeddings allow our search engine to understand the context and nuances of language, enabling it to retrieve documents that are semantically similar to the query, even if they don't contain the exact keywords. By creating embeddings using Azure OpenAI, we equip our search engine with the ability to understand and process natural language in a way that's much closer to how humans understand language. This step transforms our raw text data into a format that the search engine can use to find and rank relevant documents.



In [10]:
try:
    embeddings = AzureOpenAIEmbeddings(
        deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
        openai_api_key=AZURE_OPENAI_KEY,
        azure_endpoint=AZURE_OPENAI_ENDPOINT
    )
    # Test the embedding by making a small request
    test_embedding = embeddings.embed_query("test")
    if not test_embedding:
        raise ValueError("Embedding test failed - no embedding returned")
    logging.info("Successfully created AzureOpenAIEmbeddings and verified deployment")
except Exception as e:
    error_msg = str(e)
    if "DeploymentNotFound" in error_msg or "404" in error_msg:
        raise ValueError(
            f"Azure OpenAI deployment '{AZURE_OPENAI_EMBEDDING_DEPLOYMENT}' not found at endpoint '{AZURE_OPENAI_ENDPOINT}'. "
            f"Please verify: 1) The deployment name matches exactly (case-sensitive), "
            f"2) The deployment exists in Azure OpenAI Studio, "
            f"3) The endpoint URL is correct. Original error: {error_msg}"
        )
    raise ValueError(f"Error creating AzureOpenAIEmbeddings: {error_msg}")

2026-04-06 14:53:01,725 - INFO - HTTP Request: POST https://sample-azure-openai-key.openai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2023-05-15 "HTTP/1.1 200 OK"
2026-04-06 14:53:01,729 - INFO - Successfully created AzureOpenAIEmbeddings and verified deployment


## Setting Up the Couchbase Vector Store
The vector store is set up to manage the embeddings created in the previous step. The vector store is essentially a database optimized for storing and retrieving high-dimensional vectors. In this case, the vector store is built on top of Couchbase, allowing the script to store the embeddings in a way that can be efficiently searched.

In [11]:
try:
    vector_store = CouchbaseSearchVectorStore(
        cluster=cluster,
        bucket_name=CB_BUCKET_NAME,
        scope_name=SCOPE_NAME,
        collection_name=COLLECTION_NAME,
        embedding=embeddings,
        index_name=INDEX_NAME,
    )
    logging.info("Successfully created vector store")
except Exception as e:
    raise ValueError(f"Failed to create vector store: {str(e)}")

2026-04-06 14:57:00,169 - INFO - Successfully created vector store


## Saving Data to the Vector Store
With the vector store set up, the next step is to populate it with data. We save the BBC News dataset to the vector store in batches. This method is efficient and ensures that our search engine can handle large datasets without running into performance issues. By saving the data in this way, we prepare our search engine to quickly and accurately respond to user queries. This step is essential for making the dataset searchable, transforming raw data into a format that can be easily queried by our search engine.

Batch processing is particularly important when dealing with large datasets, as it prevents memory overload and ensures that the data is stored in a structured and retrievable manner. This approach not only optimizes performance but also ensures the scalability of our system.

In [12]:
try:
    batch_size = 50
    logging.disable(sys.maxsize) # Disable logging to prevent tqdm output
    for i in tqdm(range(0, len(unique_news_articles), batch_size), desc="Processing Batches"):
        batch = unique_news_articles[i:i + batch_size]
        documents = [Document(page_content=text) for text in batch]
        uuids = [str(uuid4()) for _ in range(len(documents))]
        vector_store.add_documents(documents=documents, ids=uuids)
    logging.disable(logging.NOTSET) # Re-enable logging
except Exception as e:
    raise RuntimeError(f"Failed to save documents to vector store: {str(e)}")

Processing Batches: 100%|██████████| 35/35 [05:20<00:00,  9.16s/it]


## Setting Up a Couchbase Cache
To further optimize our system, we set up a Couchbase-based cache. A cache is a temporary storage layer that holds data that is frequently accessed, speeding up operations by reducing the need to repeatedly retrieve the same information from the database. In our setup, the cache will help us accelerate repetitive tasks, such as looking up similar documents. By implementing a cache, we enhance the overall performance of our search engine, ensuring that it can handle high query volumes and deliver results quickly.

Caching is particularly valuable in scenarios where users may submit similar queries multiple times or where certain pieces of information are frequently requested. By storing these in a cache, we can significantly reduce the time it takes to respond to these queries, improving the user experience.


In [13]:
try:
    cache = CouchbaseCache(
        cluster=cluster,
        bucket_name=CB_BUCKET_NAME,
        scope_name=SCOPE_NAME,
        collection_name=CACHE_COLLECTION,
    )
    logging.info("Successfully created cache")
    set_llm_cache(cache)
except Exception as e:
    raise ValueError(f"Failed to create cache: {str(e)}")

2026-04-06 15:06:49,385 - INFO - Successfully created cache


## Using the Azure Chat OpenAI Language Model (LLM)
Language models are AI systems that are trained to understand and generate human language. We'll be using `AzureChatOpenAI` language model to process user queries and generate meaningful responses. This model is a key component of our semantic search engine, allowing it to go beyond simple keyword matching and truly understand the intent behind a query. By creating this language model, we equip our search engine with the ability to interpret complex queries, understand the nuances of language, and provide more accurate and contextually relevant responses.

The language model's ability to understand context and generate coherent responses is what makes our search engine truly intelligent. It can not only find the right information but also present it in a way that is useful and understandable to the user.



In [14]:
try:
    llm = AzureChatOpenAI(
        deployment_name=AZURE_OPENAI_CHAT_DEPLOYMENT,
        openai_api_key=AZURE_OPENAI_KEY,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        openai_api_version="2024-07-01-preview"
    )
    # Test the LLM by making a small request
    test_response = llm.invoke("Say 'test successful'")
    if not test_response:
        raise ValueError("LLM test failed - no response returned")
    logging.info("Successfully created Azure OpenAI Chat model and verified deployment")
except Exception as e:
    error_msg = str(e)
    if "DeploymentNotFound" in error_msg or "404" in error_msg:
        raise ValueError(
            f"Azure OpenAI deployment '{AZURE_OPENAI_CHAT_DEPLOYMENT}' not found at endpoint '{AZURE_OPENAI_ENDPOINT}'. "
            f"Please verify: 1) The deployment name matches exactly (case-sensitive), "
            f"2) The deployment exists in Azure OpenAI Studio, "
            f"3) The endpoint URL is correct. Original error: {error_msg}"
        )
    raise ValueError(f"Error creating Azure OpenAI Chat model: {error_msg}")

2026-04-06 15:06:52,202 - INFO - HTTP Request: POST https://sample-azure-openai-key.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-07-01-preview "HTTP/1.1 200 OK"
2026-04-06 15:06:52,212 - INFO - Successfully created Azure OpenAI Chat model and verified deployment


## Perform Semantic Search
Semantic search in Couchbase involves converting queries and documents into vector representations using an embeddings model. These vectors capture the semantic meaning of the text and are stored directly in Couchbase. When a query is made, Couchbase performs a similarity search by comparing the query vector against the stored document vectors. The similarity metric used for this comparison is configurable, allowing flexibility in how the relevance of documents is determined. Common metrics include cosine similarity, Euclidean distance, or dot product, but other metrics can be implemented based on specific use cases. Different embedding models like BERT, Word2Vec, or GloVe can also be used depending on the application's needs, with the vectors generated by these models stored and searched within Couchbase itself.

In the provided code, the search process begins by recording the start time, followed by executing the similarity_search_with_score method of the CouchbaseSearchVectorStore. This method searches Couchbase for the most relevant documents based on the vector similarity to the query. The search results include the document content and a similarity score that reflects how closely each document aligns with the query in the defined semantic space. The time taken to perform this search is then calculated and logged, and the results are displayed, showing the most relevant documents along with their similarity scores. This approach leverages Couchbase as both a storage and retrieval engine for vector data, enabling efficient and scalable semantic searches. The integration of vector storage and search capabilities within Couchbase allows for sophisticated semantic search operations without relying on external services for vector storage or comparison.

In [15]:
query = "What caused the 1929 Great Depression?"

def perform_search():
    """Wrapper function for semantic search to enable retry logic."""
    start_time = time.time()
    results = vector_store.similarity_search_with_score(query, k=10)
    elapsed = time.time() - start_time
    return results, elapsed

try:
    # Perform the semantic search with retry logic
    search_results, search_elapsed_time = retry_with_backoff(perform_search)

    logging.info(f"Semantic search completed in {search_elapsed_time:.2f} seconds")

    # Display search results
    print(f"\nSemantic Search Results (completed in {search_elapsed_time:.2f} seconds):")
    for doc, score in search_results:
        print(f"Distance: {score:.4f}, Text: {doc.page_content}")

except InternalServerFailureException as e:
    if "query request rejected" in str(e):
        raise RuntimeError("Search request was rate limited after multiple retries. Please wait and try again later.")
    raise RuntimeError(f"Error performing semantic search: {str(e)}")
except CouchbaseException as e:
    raise RuntimeError(f"Error performing semantic search: {str(e)}")
except Exception as e:
    raise RuntimeError(f"Unexpected error: {str(e)}")

2026-04-06 15:06:55,740 - INFO - HTTP Request: POST https://sample-azure-openai-key.openai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2023-05-15 "HTTP/1.1 200 OK"
2026-04-06 15:06:55,770 - INFO - Semantic search completed in 1.05 seconds



Semantic Search Results (completed in 1.05 seconds):
Distance: 0.2631, Text: Stocks slide as US central bank signals slower pace of rate cuts

US share prices slumped after the central bank cut interest rates for the third time in a row but its economic projections signalled a slower pace of cuts next year. In a widely expected move, the Federal Reserve set its key lending rate in a target range of 4.25% to 4.5%. That is down a full percentage point since September, when the bank started lowering borrowing costs, citing progress stabilising prices and a desire to head off economic weakening. Reports since then indicate that the number of jobs being created has been more resilient than expected, while price rises have continued to bubble.

Stocks in the US fell sharply as Federal Reserve chairman Jerome Powell warned the situation would likely result in fewer rate cuts than expected next year. "We are in a new phase of the process," he said at a press conference. "From this point forwa

## Retrieval-Augmented Generation (RAG) with Couchbase and Langchain
Couchbase and LangChain can be seamlessly integrated to create RAG (Retrieval-Augmented Generation) chains, enhancing the process of generating contextually relevant responses. In this setup, Couchbase serves as the vector store, where embeddings of documents are stored. When a query is made, LangChain retrieves the most relevant documents from Couchbase by comparing the query’s embedding with the stored document embeddings. These documents, which provide contextual information, are then passed to a generative language model within LangChain.

The language model, equipped with the context from the retrieved documents, generates a response that is both informed and contextually accurate. This integration allows the RAG chain to leverage Couchbase’s efficient storage and retrieval capabilities, while LangChain handles the generation of responses based on the context provided by the retrieved documents. Together, they create a powerful system that can deliver highly relevant and accurate answers by combining the strengths of both retrieval and generation.

In [16]:
template = """You are a helpful bot. If you cannot answer based on the context provided, respond with a generic answer. Answer the question as truthfully as possible using the context below:
    {context}
    Question: {question}"""
prompt = ChatPromptTemplate.from_template(template)
rag_chain = (
    {"context": vector_store.as_retriever(), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
logging.info("Successfully created RAG chain")

2026-04-06 15:07:01,582 - INFO - Successfully created RAG chain


In [17]:
# Get responses
logging.disable(sys.maxsize) # Disable logging to prevent tqdm output
start_time = time.time()
rag_response = rag_chain.invoke(query)
rag_elapsed_time = time.time() - start_time

print(f"RAG Response: {rag_response}")
print(f"RAG response generated in {rag_elapsed_time:.2f} seconds")

RAG Response: The 1929 Great Depression was primarily caused by the stock market crash of October 1929, which was preceded by a speculative bubble where stock prices soared to unsustainable levels. Contributing factors included excessive stock market speculation, a lack of regulation in the financial sector, an unequal distribution of wealth, and economic policies that favored large businesses and banks. Additionally, widespread bank failures, reductions in consumer spending, and a restrictive monetary policy by the Federal Reserve exacerbated the economic downturn. The combination of these elements led to severe economic decline, high unemployment rates, and a prolonged period of hardship in the following years.
RAG response generated in 3.78 seconds


## Using Couchbase as a caching mechanism
Couchbase can be effectively used as a caching mechanism for RAG (Retrieval-Augmented Generation) responses by storing and retrieving precomputed results for specific queries. This approach enhances the system's efficiency and speed, particularly when dealing with repeated or similar queries. When a query is first processed, the RAG chain retrieves relevant documents, generates a response using the language model, and then stores this response in Couchbase, with the query serving as the key.

For subsequent requests with the same query, the system checks Couchbase first. If a cached response is found, it is retrieved directly from Couchbase, bypassing the need to re-run the entire RAG process. This significantly reduces response time because the computationally expensive steps of document retrieval and response generation are skipped. Couchbase's role in this setup is to provide a fast and scalable storage solution for caching these responses, ensuring that frequently asked queries can be answered more quickly and efficiently.


In [18]:
try:
    queries = [
        "Why do heavier objects travel downhill faster?",
        "What is the capital of France?",
        "What caused the 1929 Great Depression?", # Repeated query
        "Why do heavier objects travel downhill faster?",  # Repeated query
    ]

    for i, query in enumerate(queries, 1):
        print(f"\nQuery {i}: {query}")
        start_time = time.time()

        def perform_rag_query():
            """Wrapper for RAG query execution with retry logic."""
            return rag_chain.invoke(query)

        response = retry_with_backoff(perform_rag_query)
        elapsed_time = time.time() - start_time
        print(f"Response: {response}")
        print(f"Time taken: {elapsed_time:.2f} seconds")

        # Add delay between queries to avoid rate limiting
        if i < len(queries):
            time.sleep(0.5)

except InternalServerFailureException as e:
    if "query request rejected" in str(e):
        raise ValueError("Search request was rate limited after multiple retries. Please wait and try again later.")
    raise ValueError(f"Error generating RAG response: {str(e)}")
except Exception as e:
    raise ValueError(f"Error generating RAG response: {str(e)}")


Query 1: Why do heavier objects travel downhill faster?
Response: Heavier objects do not necessarily travel downhill faster than lighter ones when air resistance is not a factor. In a vacuum, all objects fall at the same rate regardless of their weight, according to Galileo's principle of falling bodies. However, in the presence of air resistance, this can change: lighter objects may be more affected by air resistance, causing them to fall slower than heavier objects.
Time taken: 1.48 seconds

Query 2: What is the capital of France?
Response: The capital of France is Paris.
Time taken: 0.95 seconds

Query 3: What caused the 1929 Great Depression?
Response: The 1929 Great Depression was primarily caused by the stock market crash of October 1929, which was preceded by a speculative bubble where stock prices soared to unsustainable levels. Contributing factors included excessive stock market speculation, a lack of regulation in the financial sector, an unequal distribution of wealth, and

## Conclusion

You've built a semantic search engine using Couchbase Search Vector Index with Azure OpenAI embeddings and LangChain. For the Hyperscale or Composite Vector Index alternative, see the [query_based tutorial](https://developer.couchbase.com/tutorial-azure-openai-couchbase-rag-with-hyperscale-or-composite-vector-index).